# Time Series Forecasting for Real-World Data

This notebook demonstrates a complete forecasting workflow for industrial-style time series data using Python.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from time_series_forecasting.data_loader import generate_sample_data, load_dataset
from time_series_forecasting.preprocessing import prepare_features
from time_series_forecasting.models import fit_arima_model, fit_linear_regression_model, forecast_arima_model, forecast_linear_regression_model, evaluate_forecast
from time_series_forecasting.visualization import plot_time_series, plot_actual_vs_predicted, plot_residuals, plot_trend_analysis

In [ ]:
data_path = ROOT / "data" / "industrial_energy.csv"
if not data_path.exists():
    generate_sample_data(data_path)

df = load_dataset(data_path)
df.head()

In [ ]:
prepared_df = prepare_features(df)
prepared_df.head()

In [ ]:
horizon = 30
train_df = prepared_df.iloc[:-horizon]
test_df = prepared_df.iloc[-horizon:]

feature_columns = [column for column in train_df.columns if column not in {"date", "value"}]
X_train = train_df[feature_columns]
y_train = train_df["value"]

arima_model = fit_arima_model(train_df["value"], order=(2, 0, 2))
arima_forecast = forecast_arima_model(arima_model, horizon)

linear_model = fit_linear_regression_model(X_train, y_train)
linear_forecast = forecast_linear_regression_model(
    model=linear_model,
    history=list(train_df["value"].tolist()),
    future_dates=list(test_df["date"].tolist()),
    feature_columns=feature_columns,
)

arima_metrics = evaluate_forecast(test_df["value"].tolist(), arima_forecast.tolist())
linear_metrics = evaluate_forecast(test_df["value"].tolist(), linear_forecast)

arima_metrics, linear_metrics

In [ ]:
plot_time_series(df, save_path=ROOT / "outputs" / "time_series.png")
plot_trend_analysis(df, save_path=ROOT / "outputs" / "trend_analysis.png")
plot_actual_vs_predicted(test_df["value"].tolist(), arima_forecast.tolist(), "ARIMA", save_path=ROOT / "outputs" / "arima_vs_actual.png")
plot_actual_vs_predicted(test_df["value"].tolist(), linear_forecast, "Linear Regression", save_path=ROOT / "outputs" / "linear_regression_vs_actual.png")
plot_residuals(test_df["value"].tolist(), linear_forecast, save_path=ROOT / "outputs" / "residuals.png")